# PlayStation Game Success Analysis

This notebook is the starting point for the project proposed in the screenshots.
It covers:

- loading and cleaning the dataset
- checking missing values and structure
- exploring a few core relationships
- training the first baseline prediction model


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from playstation_success.data import load_clean_dataset
from playstation_success.eda import build_eda_tables
from playstation_success.features import build_feature_frame
from playstation_success.model import train_baseline_model

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


## Load The Dataset

In [ ]:
df = load_clean_dataset()
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)

## Missing Values And Data Quality

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename('missing_count')
    .to_frame()
    .assign(missing_pct=lambda frame: (frame['missing_count'] / len(df) * 100).round(2))
    .sort_values('missing_count', ascending=False)
)
missing_summary

In [ ]:
df.describe(include='all').transpose().head(15)

## EDA Tables

In [ ]:
eda_tables = build_eda_tables(df)
eda_tables['platform_summary'].head(10)

In [ ]:
eda_tables['genre_summary'].head(10)

In [ ]:
eda_tables['correlations']

## Visual Exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['highest_price'].dropna(), bins=30, ax=axes[0], color='#1f77b4')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Highest Price')

score_frame = df[['metacritic_score', 'playstation_score']].dropna()
sns.scatterplot(
    data=score_frame,
    x='metacritic_score',
    y='playstation_score',
    alpha=0.45,
    ax=axes[1],
)
axes[1].set_title('Metacritic Score vs PlayStation Score')

plt.tight_layout()
plt.show()

In [ ]:
feature_df = build_feature_frame(df)
platform_plot = (
    feature_df.groupby('platform')
    .agg(avg_playstation_score=('playstation_score', 'mean'), game_count=('game_name', 'count'))
    .query('game_count >= 25')
    .sort_values('avg_playstation_score', ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.barplot(data=platform_plot, x='platform', y='avg_playstation_score', palette='Blues_d')
plt.title('Average PlayStation Score by Platform')
plt.xlabel('Platform')
plt.ylabel('Average PlayStation Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Baseline Model

In [ ]:
training_result = train_baseline_model(df)
pd.DataFrame([training_result.metrics])

In [ ]:
training_result.feature_importance.head(15)

In [ ]:
top_features = training_result.feature_importance.head(12).sort_values('importance')

plt.figure(figsize=(10, 7))
plt.barh(top_features['feature'], top_features['importance'], color='#2a6f97')
plt.title('Top Baseline Model Features')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
training_result.predictions.head(10)

## Next Steps

1. Define a stronger target for "success" such as a high-score or high-engagement label.
2. Compare regression and classification approaches.
3. Turn the final results into a presentation notebook or dashboard.
